In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from umap import UMAP
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from pycocotools import mask as mask_utils
import os
import json
import torchvision.models as models
import torch.nn as nn
import torch

In [2]:
COCO_JSON = "../data/Cerrado_trees.v1i.coco/train/_annotations.coco.json"
IMAGE_ROOT = "../data/Cerrado_trees.v1i.coco/train"
OUT_DIR = "../data/Cerrado_trees.v1i.coco/_separated"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# LOAD COCO
# -----------------------
with open(COCO_JSON, "r") as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]}

# Build category_id → name mapping
cat_id_to_name = {cat["id"]: cat["name"] for cat in coco["categories"]}

# group annotations per image
annos_by_image = {}
for ann in coco["annotations"]:
    iid = ann["image_id"]
    if iid not in annos_by_image:
        annos_by_image[iid] = []
    annos_by_image[iid].append(ann)

# -----------------------
# MAIN LOOP
# -----------------------
for image_id, anns in tqdm(annos_by_image.items()):
    img_info = images[image_id]
    img_path = os.path.join(IMAGE_ROOT, img_info["file_name"])
    image = np.array(Image.open(img_path).convert("RGB"))

    for idx, ann in enumerate(anns):
        # ---- get mask from polygon or RLE
        seg = ann["segmentation"]
        
        if isinstance(seg, list):
            rles = mask_utils.frPyObjects(seg, img_info["height"], img_info["width"])
            rle = mask_utils.merge(rles)
        else:
            rle = seg
        
        mask = mask_utils.decode(rle)  # HxW binary mask

        # ---- extract object pixels
        obj = image.copy()
        obj[mask == 0] = 255  # white background

        # ---- tight bounding box
        ys, xs = np.where(mask == 1)
        y1, y2 = ys.min(), ys.max()
        x1, x2 = xs.min(), xs.max()
        cropped = obj[y1:y2+1, x1:x2+1]

        # ---- get label name
        cat_id = ann["category_id"]
        label = cat_id_to_name.get(cat_id, f"id{cat_id}")

        # safe filename (no spaces)
        label = label.replace(" ", "_")

        # ---- save
        base = img_info['file_name'].split('.')[0]
        out_name = f"{base}_{label}_{idx+1}.png"
        out_path = os.path.join(OUT_DIR, out_name)

        Image.fromarray(cropped).save(out_path)

print("Done! Extracted all object crops with label names.")

100%|██████████████████████████████████████████████████████████████████████████████████| 54/54 [03:03<00:00,  3.39s/it]

Done! Extracted all object crops with label names.


In [4]:
class TreeCrownResNet34(nn.Module):
    """Fine-tuned ResNet34 for 512x512 tree crown masks"""

    def __init__(self, freeze_backbone=True, latent_dim=256, use_pretrained=True):
        super().__init__()
        
        # Load pretrained weights or random init
        weights = ResNet34_Weights.IMAGENET1K_V1 if use_pretrained else None
        self.backbone = models.resnet34(weights=weights)

        # Grab feature dimension before fc
        in_features = self.backbone.fc.in_features
        
        # Remove the final classifier head
        self.backbone.fc = nn.Identity()
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, latent_dim)
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, in_features)
        )

        # Store preprocessing from weights (for transforms)
        self.preprocess = weights.transforms() if use_pretrained else None

    def encode(self, x):
        features = self.backbone(x)   # (B, 512)
        return self.projection(features)

    def forward(self, x):
        features = self.backbone(x)
        latent = self.projection(features)
        reconstructed = self.decoder(latent)
        return latent, reconstructed, features

In [7]:
model = TreeCrownResNet34(use_pretrained=False)
model.load_state_dict(torch.load("resnet34.pth"))

<All keys matched successfully>